In [4]:
!pip install -q -U transformers
!pip install -q -U datasets
!pip install -q -U bitsandbytes
!pip install -q -U trl
!pip install pylatexenc sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 88.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 17.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cupti-cu12 12

In [5]:
from huggingface_hub import login
login()

In [6]:
from transformers import (pipeline, AutoModelForCausalLM, AutoTokenizer,
                          BitsAndBytesConfig, TrainingArguments, Trainer)
from datasets import load_dataset, Dataset
import bitsandbytes as bnb
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
import torch

import numpy as np
import random
import re
from pprint import pprint

from google.colab import drive, userdata

In [7]:
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [8]:
project_path = "/content/drive/MyDrive/Final Project"

In [9]:
import sys

In [10]:
sys.path.append('/content/drive/MyDrive/Final Project/src/grading')

In [17]:
#from grader import grade_answer_with_label

Dataset Loading

In [14]:
Arithmetics = load_dataset("json", data_files=f"{project_path}/data/arithmetics.json")
print("Question:", Arithmetics['train']['question'][0])
print("Answer:", Arithmetics['train']['answer'][0])

Question: 9-0+14+18+15*23=?
Answer: 386


In [11]:
GSM= load_dataset("json", data_files=f"{project_path}/data/gsm.json")
print("Question:", GSM['train']['question'][0])
print("Answer:", GSM['train']['answer'][0])

Generating train split: 0 examples [00:00, ? examples/s]

Question: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?
Answer: Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18


In [17]:
HSM= load_dataset("json", data_files=f"{project_path}/data/hsm.json")
print("Question:", HSM['train']['problem'][0])
print("Answer:", HSM['train']['answer'][0])

Question: Convert the point $(0,3)$ in rectangular coordinates to polar coordinates.  Enter your answer in the form $(r,\theta),$ where $r > 0$ and $0 \le \theta < 2 \pi.$
Answer: \left( 3, \frac{\pi}{2} \right)


In [19]:
ThmQA= load_dataset("json", data_files=f"{project_path}/data/thmqa.json")
print("Question:", ThmQA['train']['Question'][0])
print("Answer:", ThmQA['train']['Answer'][0])

Question: How many ways are there to divide a set of 8 elements into 5 non-empty ordered subsets?
Answer: 11760


Prompts

In [14]:
def make_base_prompt(question):
  return f"""You are a helpful assistant. Provide the answer to the question below as \\boxed{{answer}} at the end. \n{question}"""

In [15]:
def make_cot_prompt(question):
  return f"""You are a helpful assistant. Solve the question below and provide the answer as \\boxed{{answer}} at the end. Let's think step by step. \n{question}"""

In [16]:
def make_qap_prompt(question):
    return f"""You are a helpful assistant.  Answer the following question directly and clearly. Show any necessary work briefly in under 50 words, then provide the final answer.\n\nQuestion: {question}\n\nAnswer: \\boxed{{answer}}"""

In [17]:
def make_mad_prompt(question):
    import re

    agents = [
        "Mathematician",
        "Counterexample Expert",
        "Computational Mathematician",
        "Theoretician"
    ]

    basic_arithmetic = re.match(r"^\d+\s*[\+\-\*/]\s*\d+$", question.strip())

    if basic_arithmetic:
        return (
            f"You are a panel of experts solving this arithmetic problem: \"{question}\"\n\n"
            "Each expert briefly shares their answer. Then a final consensus answer is provided.\n\n"
            "## Final Answer:\n"
            "Please conclude with the final result in this format: \\boxed{your_answer}"
        )

    if "prove" in question.lower():
        debate_topics = [
            "Proof Strategy",
            "Logical Reasoning",
            "Edge Cases",
            "Theoretical Soundness"
        ]
    elif "divisible" in question.lower():
        debate_topics = [
            "Divisibility Rules",
            "Prime Properties",
            "Simplification",
            "Counterexamples"
        ]
    else:
        debate_topics = [
            "General Approach",
            "Computational Tools",
            "Abstract Theory",
            "Verification"
        ]

    prompt = f"""You are participating in a multi-agent debate to solve the following problem:

"{question}"

Each expert presents a concise viewpoint. After all opinions are shared, a final consensus is formed.

Participants:"""

    for i, agent in enumerate(agents, start=1):
        prompt += f"\n{i}. **{agent}**: Provides their viewpoint."

    prompt += "\n\nDiscussion Topics:\n"
    for topic in debate_topics:
        prompt += f"- {topic}\n"

    prompt += """

Simulate the discussion among the experts with each agent providing under 20 words at most. Then provide a clear final answer summary.

## Final Consensus:
Summarize the reasoning briefly under 50 words, then clearly state the final boxed answer below.

Final Answer: \\boxed{...}
"""

    return prompt


In [18]:
# Example usage with a basic question like "1 + 1":
question = "Why does this not work?"
debate_prompt = make_mad_prompt(question)

# Output the result
print(debate_prompt)

You are participating in a multi-agent debate to solve the following problem:

"Why does this not work?"

Each expert presents a concise viewpoint. After all opinions are shared, a final consensus is formed.

Participants:
1. **Mathematician**: Provides their viewpoint.
2. **Counterexample Expert**: Provides their viewpoint.
3. **Computational Mathematician**: Provides their viewpoint.
4. **Theoretician**: Provides their viewpoint.

Discussion Topics:
- General Approach
- Computational Tools
- Abstract Theory
- Verification


Simulate the discussion among the experts with each agent providing under 20 words at most. Then provide a clear final answer summary.

## Final Consensus:
Summarize the reasoning briefly under 50 words, then clearly state the final boxed answer below.

Final Answer: \boxed{...}



In [19]:
def generate_answers(questions, prompt_fn, model_pipe, max_new_tokens=512):
    predictions = []
    for q in questions:
        prompt = prompt_fn(q)
        output = model_pipe(prompt, max_new_tokens=max_new_tokens, do_sample=False)[0]["generated_text"]

        # Safely remove the prompt only if it's a clean match (prefix or appears early)
        if prompt in output:
            # If prompt appears early, trim before or after it
            idx = output.find(prompt) + len(prompt)
            generated = output[idx:].strip()
        else:
            generated = output.strip()

        predictions.append(generated)
    return predictions


Evaluation Criteria

Models

In [20]:
"""
Initialize the pipeline with bitsandbytes quantization
"""
# Configure bitsandbytes for 4-bit quantization
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [29]:
# Load Llama Model
model_id = "meta-llama/Llama-3.2-3B-Instruct"

Llama_pipe = pipeline(
   "text-generation",
   model=model_id,
   model_kwargs={"torch_dtype": torch.bfloat16, "quantization_config": quantization_config},
   device_map="auto",
   trust_remote_code=True
)

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Device set to use cuda:0


In [21]:
# Load Gemma Model
model_id = "google/gemma-3-4b-it"

Gemma_pipe = pipeline(
   "text-generation",
   model=model_id,
   model_kwargs={"torch_dtype": torch.bfloat16, "quantization_config": quantization_config},
   device_map="auto",
   trust_remote_code=True
)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.64G [00:00<?, ?B/s]

...

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

Device set to use cuda:0


Prompting LLamma

In [134]:
questions = Arithmetics['train']['question'][:100]
gold_answers = Arithmetics['train']['answer'][:100]
#Run Base Prompt
base_preds = generate_answers(questions, make_base_prompt, Llama_pipe)

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_

In [128]:
questions = Arithmetics['train']['question'][:100]
gold_answers = Arithmetics['train']['answer'][:100]

# Run CoT
cot_preds = generate_answers(questions, make_cot_prompt, Llama_pipe)
# Run QAP
qap_preds = generate_answers(questions, make_qap_prompt, Llama_pipe)
# Run MAD
mad_preds = generate_answers(questions, make_mad_prompt, Llama_pipe)

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_

In [135]:
base_results = [grade_answer_with_label(pred, str(gold)) for pred, gold in zip(base_preds, gold_answers)]

In [129]:

# Evaluate all
cot_results = [grade_answer_with_label(pred, str(gold)) for pred, gold in zip(cot_preds, gold_answers)]
qap_results = [grade_answer_with_label(pred, str(gold)) for pred, gold in zip(qap_preds, gold_answers)]
mad_results = [grade_answer_with_label(pred, str(gold)) for pred, gold in zip(mad_preds, gold_answers)]

In [137]:

base_df = pd.DataFrame({
    "method": "Baseline",
    "question": questions,
    "gold": gold_answers,
    "pred": base_preds,
    "eval": base_results
})

cot_df = pd.DataFrame({
    "method": "CoT",
    "question": questions,
    "gold": gold_answers,
    "pred": cot_preds,
    "eval": cot_results
})

qap_df = pd.DataFrame({
    "method": "QAP",
    "question": questions,
    "gold": gold_answers,
    "pred": qap_preds,
    "eval": qap_results
})

mad_df = pd.DataFrame({
    "method": "MAD",
    "question": questions,
    "gold": gold_answers,
    "pred": mad_preds,
    "eval": mad_results
})
all_results = pd.concat([base_df,cot_df, qap_df, mad_df])
accuracy_summary = all_results.groupby("method")["eval"].value_counts(normalize=True).unstack().fillna(0)
from IPython.display import display, Markdown

display(Markdown("### 🔍 Prompting Llamma with Arithmetics problems"))
display(accuracy_summary)


### 🔍 Prompting Llamma with Arithmetics problems

eval,correct_mathd,correct_semantic,incorrect
method,,,
Baseline,0.78,0.04,0.18
CoT,0.77,0.04,0.19
MAD,0.01,0.00,0.99
QAP,0.55,0.03,0.42


In [131]:
from IPython.display import display, Markdown

display(Markdown("### 🔍 Prompting Llamma with Arithmetics problems"))
display(accuracy_summary)


### 🔍 Prompting Llamma with Arithmetics problems

eval,correct_mathd,correct_semantic,incorrect
method,,,
CoT,0.77,0.04,0.19
MAD,0.01,0.00,0.99
QAP,0.55,0.03,0.42


In [163]:
questions = GSM['train']['question'][:100]
gold_answers = GSM['train']['answer'][:100]

#Run Base Prompt
base_preds = generate_answers(questions, make_base_prompt, Llama_pipe)
with open('/content/drive/MyDrive/Final Project/src/Results/llamma_gsm_base_preds.json', 'w') as f:
    json.dump(base_preds, f)
# Run CoT
cot_preds = generate_answers(questions, make_cot_prompt, Llama_pipe)
with open('/content/drive/MyDrive/Final Project/src/Results/llamma_gsm_cot_preds.json', 'w') as f:
    json.dump(cot_preds, f)
# Run QAP
qap_preds = generate_answers(questions, make_qap_prompt, Llama_pipe)
with open('/content/drive/MyDrive/Final Project/src/Results/llamma_gsm_qap_preds.json', 'w') as f:
    json.dump(qap_preds, f)
# Run MAD
mad_preds = generate_answers(questions, make_mad_prompt, Llama_pipe)
with open('/content/drive/MyDrive/Final Project/src/Results/llamma_gsm_mad_preds.json', 'w') as f:
    json.dump(mad_preds, f)


/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_

In [168]:

# Evaluate all
base_results = [grade_answer_with_label(pred, gold) for pred, gold in zip(base_preds, gold_answers)]
cot_results = [grade_answer_with_label(pred, gold) for pred, gold in zip(cot_preds, gold_answers)]
qap_results = [grade_answer_with_label(pred, gold) for pred, gold in zip(qap_preds, gold_answers)]
mad_results = [grade_answer_with_label(pred, gold) for pred, gold in zip(mad_preds, gold_answers)]

In [169]:


base_df = pd.DataFrame({
    "method": "CoT",
    "question": questions,
    "gold": gold_answers,
    "pred": base_preds,
    "eval": base_results
})

cot_df = pd.DataFrame({
    "method": "CoT",
    "question": questions,
    "gold": gold_answers,
    "pred": cot_preds,
    "eval": cot_results
})

qap_df = pd.DataFrame({
    "method": "QAP",
    "question": questions,
    "gold": gold_answers,
    "pred": qap_preds,
    "eval": qap_results
})

mad_df = pd.DataFrame({
    "method": "MAD",
    "question": questions,
    "gold": gold_answers,
    "pred": mad_preds,
    "eval": mad_results
})
all_results = pd.concat([base_df,cot_df, qap_df, mad_df])
accuracy_summary = all_results.groupby("method")["eval"].value_counts(normalize=True).unstack().fillna(0)
accuracy_summary

eval,incorrect
method,
CoT,1.0
MAD,1.0
QAP,1.0


In [138]:
questions = HSM['train']['problem'][:100]
gold_answers = HSM['train']['answer'][:100]

# Run Baseline
base_preds = generate_answers(questions, make_base_prompt, Llama_pipe)
# Run CoT
cot_preds = generate_answers(questions, make_cot_prompt, Llama_pipe)
# Run QAP
qap_preds = generate_answers(questions, make_qap_prompt, Llama_pipe)
# Run MAD
mad_preds = generate_answers(questions, make_mad_prompt, Llama_pipe)



/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_

KeyboardInterrupt: 

In [146]:
import json

with open('/content/drive/MyDrive/Final Project/src/Results/llamma_hsm_base_preds.json', 'w') as f:
    json.dump(base_preds, f)


In [148]:
# Run CoT
cot_preds = generate_answers(questions, make_cot_prompt, Llama_pipe)
with open('/content/drive/MyDrive/Final Project/src/Results/llamma_hsm_cot_preds.json', 'w') as f:
    json.dump(cot_preds, f)

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_

In [149]:
# Run QAP
qap_preds = generate_answers(questions, make_qap_prompt, Llama_pipe)
with open('/content/drive/MyDrive/Final Project/src/Results/llamma_hsm_qap_preds.json', 'w') as f:
    json.dump(qap_preds, f)

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_

In [150]:
# Run MAD
mad_preds = generate_answers(questions, make_mad_prompt, Llama_pipe)
with open('/content/drive/MyDrive/Final Project/src/Results/llamma_hsm_mad_preds.json', 'w') as f:
    json.dump(mad_preds, f)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

In [151]:
# Evaluate all
base_results = [grade_answer_with_label(pred, gold) for pred, gold in zip(base_preds, gold_answers)]
cot_results = [grade_answer_with_label(pred, gold) for pred, gold in zip(cot_preds, gold_answers)]
qap_results = [grade_answer_with_label(pred, gold) for pred, gold in zip(qap_preds, gold_answers)]
mad_results = [grade_answer_with_label(pred, gold) for pred, gold in zip(mad_preds, gold_answers)]

In [152]:
base_df = pd.DataFrame({
    "method": "Baseline",
    "question": questions,
    "gold": gold_answers,
    "pred": base_preds,
    "eval": base_results
})

cot_df = pd.DataFrame({
    "method": "CoT",
    "question": questions,
    "gold": gold_answers,
    "pred": cot_preds,
    "eval": cot_results
})

qap_df = pd.DataFrame({
    "method": "QAP",
    "question": questions,
    "gold": gold_answers,
    "pred": qap_preds,
    "eval": qap_results
})

mad_df = pd.DataFrame({
    "method": "MAD",
    "question": questions,
    "gold": gold_answers,
    "pred": mad_preds,
    "eval": mad_results
})
all_results = pd.concat([base_df,cot_df, qap_df, mad_df])
accuracy_summary = all_results.groupby("method")["eval"].value_counts(normalize=True).unstack().fillna(0)
from IPython.display import display, Markdown

display(Markdown("### 🔍 Prompting Llamma with HSM problems"))
display(accuracy_summary)

### 🔍 Prompting Llamma with HSM problems

eval,correct_mathd,correct_semantic,correct_string,correct_symbolic,incorrect
method,,,,,
Baseline,0.30,0.05,0.03,0.01,0.61
CoT,0.36,0.04,0.04,0.00,0.56
MAD,0.10,0.05,0.02,0.01,0.82
QAP,0.21,0.05,0.02,0.00,0.72


In [153]:
questions = ThmQA['train']['Question'][:100]
gold_answers = ThmQA['train']['Answer'][:100]
# Run Baseline
base_preds = generate_answers(questions, make_base_prompt, Llama_pipe)
with open('/content/drive/MyDrive/Final Project/src/Results/llamma_tmqa_base_preds.json', 'w') as f:
    json.dump(base_preds, f)
# Run CoT
cot_preds = generate_answers(questions, make_cot_prompt, Llama_pipe)
with open('/content/drive/MyDrive/Final Project/src/Results/llamma_tmqa_cot_preds.json', 'w') as f:
    json.dump(cot_preds, f)
# Run QAP
qap_preds = generate_answers(questions, make_qap_prompt, Llama_pipe)
with open('/content/drive/MyDrive/Final Project/src/Results/llamma_tmqa_qap_preds.json', 'w') as f:
    json.dump(qap_preds, f)
# Run MAD
mad_preds = generate_answers(questions, make_mad_prompt, Llama_pipe)
with open('/content/drive/MyDrive/Final Project/src/Results/llamma_tmqa_mad_preds.json', 'w') as f:
    json.dump(mad_preds, f)

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_

KeyboardInterrupt: 

In [154]:
# Run QAP
qap_preds = generate_answers(questions, make_qap_prompt, Llama_pipe)
# Run MAD
mad_preds = generate_answers(questions, make_mad_prompt, Llama_pipe)


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

In [155]:
# Evaluate all
base_results = [grade_answer_with_label(pred, gold) for pred, gold in zip(base_preds, gold_answers)]
cot_results = [grade_answer_with_label(pred, gold) for pred, gold in zip(cot_preds, gold_answers)]
qap_results = [grade_answer_with_label(pred, gold) for pred, gold in zip(qap_preds, gold_answers)]
mad_results = [grade_answer_with_label(pred, gold) for pred, gold in zip(mad_preds, gold_answers)]

In [156]:
base_df = pd.DataFrame({
    "method": "Baseline",
    "question": questions,
    "gold": gold_answers,
    "pred": base_preds,
    "eval": base_results
})

cot_df = pd.DataFrame({
    "method": "CoT",
    "question": questions,
    "gold": gold_answers,
    "pred": cot_preds,
    "eval": cot_results
})

qap_df = pd.DataFrame({
    "method": "QAP",
    "question": questions,
    "gold": gold_answers,
    "pred": qap_preds,
    "eval": qap_results
})

mad_df = pd.DataFrame({
    "method": "MAD",
    "question": questions,
    "gold": gold_answers,
    "pred": mad_preds,
    "eval": mad_results
})
all_results = pd.concat([base_df,cot_df, qap_df, mad_df])
accuracy_summary = all_results.groupby("method")["eval"].value_counts(normalize=True).unstack().fillna(0)
from IPython.display import display, Markdown

display(Markdown("### 🔍 Prompting Llamma with ThmQA problems"))
display(accuracy_summary)

### 🔍 Prompting Llamma with ThmQA problems

eval,correct_mathd,correct_semantic,correct_string,incorrect
method,,,,
Baseline,0.11,0.03,0.00,0.86
CoT,0.09,0.01,0.02,0.88
MAD,0.07,0.04,0.01,0.88
QAP,0.03,0.02,0.01,0.94


In [170]:
def show_sample_comparisons(questions, golds, preds_cot, preds_qap, preds_mad, num_samples=5):
    for i in range(num_samples):
        print(f"Example {i + 1}")
        print(f"Question: {questions[i]}")
        print(f"Ground Truth Answer: {golds[i]}")
        print(f"Chain-of-Thought Prediction: {preds_cot[i]}")
        print(f"QAP Prediction: {preds_qap[i]}")
        print(f"MAD Prediction: {preds_mad[i]}")
        print("-" * 50)

# Display 5 sample comparisons
show_sample_comparisons(questions, gold_answers, cot_preds, qap_preds, mad_preds, num_samples=5)

Example 1
Question: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?
Ground Truth Answer: Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18
Chain-of-Thought Prediction: ## Step 1: Calculate the total number of eggs laid by Janet's ducks in a day.
Janet's ducks lay 16 eggs per day.

## Step 2: Calculate the number of eggs Janet eats for breakfast every morning.
Janet eats 3 eggs for breakfast every morning.

## Step 3: Calculate the number of eggs Janet bakes for her friends every day.
Janet bakes 4 eggs for her friends every day.

## Step 4: Calculate the total number of eggs Janet consumes every day.
Total eggs consumed = eggs for breakfast + eggs for friends = 3 + 4 = 7 eggs.



Prompting Gemma

In [171]:
questions = Arithmetics['train']['question'][:100]
gold_answers = Arithmetics['train']['answer'][:100]
#Run Base Prompt
base_preds = generate_answers(questions, make_base_prompt, Gemma_pipe)
with open('/content/drive/MyDrive/Final Project/src/Results/gemma_arith_base_preds.json', 'w') as f:
    json.dump(base_preds, f)
# Run CoT
cot_preds = generate_answers(questions, make_cot_prompt, Gemma_pipe)
with open('/content/drive/MyDrive/Final Project/src/Results/gemma_arith_cot_preds.json', 'w') as f:
    json.dump(cot_preds, f)
# Run QAP
qap_preds = generate_answers(questions, make_qap_prompt, Gemma_pipe)
with open('/content/drive/MyDrive/Final Project/src/Results/gemma_arith_qap_preds.json', 'w') as f:
    json.dump(qap_preds, f)
# Run MAD
mad_preds = generate_answers(questions, make_mad_prompt, Gemma_pipe)
with open('/content/drive/MyDrive/Final Project/src/Results/gemma_arith_mad_preds.json', 'w') as f:
    json.dump(mad_preds, f)


/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `64` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


KeyboardInterrupt: 

In [159]:
# Evaluate all
base_results = [grade_answer_with_label(pred, str(gold)) for pred, gold in zip(base_preds, gold_answers)]
cot_results = [grade_answer_with_label(pred, str(gold)) for pred, gold in zip(cot_preds, gold_answers)]
qap_results = [grade_answer_with_label(pred, str(gold)) for pred, gold in zip(qap_preds, gold_answers)]
mad_results = [grade_answer_with_label(pred, str(gold)) for pred, gold in zip(mad_preds, gold_answers)]
base_df = pd.DataFrame({
    "method": "Baseline",
    "question": questions,
    "gold": gold_answers,
    "pred": base_preds,
    "eval": base_results
})

cot_df = pd.DataFrame({
    "method": "CoT",
    "question": questions,
    "gold": gold_answers,
    "pred": cot_preds,
    "eval": cot_results
})

qap_df = pd.DataFrame({
    "method": "QAP",
    "question": questions,
    "gold": gold_answers,
    "pred": qap_preds,
    "eval": qap_results
})

mad_df = pd.DataFrame({
    "method": "MAD",
    "question": questions,
    "gold": gold_answers,
    "pred": mad_preds,
    "eval": mad_results
})
all_results = pd.concat([base_df,cot_df, qap_df, mad_df])
accuracy_summary = all_results.groupby("method")["eval"].value_counts(normalize=True).unstack().fillna(0)
from IPython.display import display, Markdown

display(Markdown("### 🔍 Prompting Gemma with Arithmetics problems"))
display(accuracy_summary)

### 🔍 Prompting Gemma with Arithmetics problems

eval,correct_mathd,correct_semantic,incorrect
method,,,
Baseline,0.69,0.02,0.29
CoT,0.94,0.00,0.06
MAD,0.31,0.02,0.67
QAP,0.75,0.02,0.23


In [22]:
import json

In [29]:
with open('/content/drive/MyDrive/Final Project/src/Results/gemma_gsm_cot_preds.json', 'w') as f:
    json.dump(cot_preds, f)

In [ ]:
questions = GSM['train']['question'][:100]
gold_answers = GSM['train']['answer'][:100]
# Run MAD
mad_preds = generate_answers(questions, make_mad_prompt, Gemma_pipe)
with open('/content/drive/MyDrive/Final Project/src/Results/gemma_gsm_mad_preds.json', 'w') as f:
    json.dump(mad_preds, f)

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `64` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [27]:
questions = GSM['train']['question'][:100]
gold_answers = GSM['train']['answer'][:100]
#base_preds = generate_answers(questions, make_base_prompt, Gemma_pipe)
#with open('/content/drive/MyDrive/Final Project/src/Results/gemma_gsm_base_preds.json', 'w') as f:
    #json.dump(base_preds, f)
# Run CoT
#cot_preds = generate_answers(questions, make_cot_prompt, Gemma_pipe)
#with open('/content/drive/MyDrive/Final Project/src/Results/gemma_gsm_cot_preds.json', 'w') as f:
    #json.dump(cot_preds, f)
# Run QAP
qap_preds = generate_answers(questions, make_qap_prompt, Gemma_pipe)
with open('/content/drive/MyDrive/Final Project/src/Results/gemma_gsm_qap_preds.json', 'w') as f:
    json.dump(qap_preds, f)
# Run MAD
mad_preds = generate_answers(questions, make_mad_prompt, Gemma_pipe)
with open('/content/drive/MyDrive/Final Project/src/Results/gemma_gsm_mad_preds.json', 'w') as f:
    json.dump(mad_preds, f)
# Evaluate all
base_results = [grade_answer_with_label(pred, gold) for pred, gold in zip(base_preds, gold_answers)]
cot_results = [grade_answer_with_label(pred, gold) for pred, gold in zip(cot_preds, gold_answers)]
qap_results = [grade_answer_with_label(pred, gold) for pred, gold in zip(qap_preds, gold_answers)]
mad_results = [grade_answer_with_label(pred, gold) for pred, gold in zip(mad_preds, gold_answers)]
base_df = pd.DataFrame({
    "method": "Baseline",
    "question": questions,
    "gold": gold_answers,
    "pred": base_preds,
    "eval": base_results
})

cot_df = pd.DataFrame({
    "method": "CoT",
    "question": questions,
    "gold": gold_answers,
    "pred": cot_preds,
    "eval": cot_results
})

qap_df = pd.DataFrame({
    "method": "QAP",
    "question": questions,
    "gold": gold_answers,
    "pred": qap_preds,
    "eval": qap_results
})

mad_df = pd.DataFrame({
    "method": "MAD",
    "question": questions,
    "gold": gold_answers,
    "pred": mad_preds,
    "eval": mad_results
})
all_results = pd.concat([base_df,cot_df, qap_df, mad_df])
accuracy_summary = all_results.groupby("method")["eval"].value_counts(normalize=True).unstack().fillna(0)
from IPython.display import display, Markdown

display(Markdown("### 🔍 Prompting Gemma with GSM problems"))
display(accuracy_summary)

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `64` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


NameError: name 'json' is not defined

In [160]:
questions = HSM['train']['problem'][:100]
gold_answers = HSM['train']['answer'][:100]
base_preds = generate_answers(questions, make_base_prompt, Gemma_pipe)
with open('/content/drive/MyDrive/Final Project/src/Results/gemma_hsm_base_preds.json', 'w') as f:
    json.dump(base_preds, f)
# Run CoT
cot_preds = generate_answers(questions, make_cot_prompt, Gemma_pipe)
with open('/content/drive/MyDrive/Final Project/src/Results/gemma_hsm_cot_preds.json', 'w') as f:
    json.dump(cot_preds, f)
# Run QAP
qap_preds = generate_answers(questions, make_qap_prompt, Gemma_pipe)
with open('/content/drive/MyDrive/Final Project/src/Results/gemma_hsm_qap_preds.json', 'w') as f:
    json.dump(qap_preds, f)
# Run MAD
mad_preds = generate_answers(questions, make_mad_prompt, Gemma_pipe)
with open('/content/drive/MyDrive/Final Project/src/Results/gemma_hsm_mad_preds.json', 'w') as f:
    json.dump(mad_preds, f)
# Evaluate all
base_results = [grade_answer_with_label(pred, gold) for pred, gold in zip(base_preds, gold_answers)]
cot_results = [grade_answer_with_label(pred, gold) for pred, gold in zip(cot_preds, gold_answers)]
qap_results = [grade_answer_with_label(pred, gold) for pred, gold in zip(qap_preds, gold_answers)]
mad_results = [grade_answer_with_label(pred, gold) for pred, gold in zip(mad_preds, gold_answers)]
base_df = pd.DataFrame({
    "method": "Baseline",
    "question": questions,
    "gold": gold_answers,
    "pred": base_preds,
    "eval": base_results
})

cot_df = pd.DataFrame({
    "method": "CoT",
    "question": questions,
    "gold": gold_answers,
    "pred": cot_preds,
    "eval": cot_results
})

qap_df = pd.DataFrame({
    "method": "QAP",
    "question": questions,
    "gold": gold_answers,
    "pred": qap_preds,
    "eval": qap_results
})

mad_df = pd.DataFrame({
    "method": "MAD",
    "question": questions,
    "gold": gold_answers,
    "pred": mad_preds,
    "eval": mad_results
})
all_results = pd.concat([base_df,cot_df, qap_df, mad_df])
accuracy_summary = all_results.groupby("method")["eval"].value_counts(normalize=True).unstack().fillna(0)
from IPython.display import display, Markdown

display(Markdown("### 🔍 Prompting Gemma with HSM problems"))
display(accuracy_summary)

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `64` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


### 🔍 Prompting Gemma with HSM problems

eval,correct_mathd,correct_semantic,correct_string,incorrect
method,,,,
Baseline,0.36,0.05,0.04,0.55
CoT,0.38,0.05,0.03,0.54
MAD,0.31,0.05,0.03,0.61
QAP,0.33,0.01,0.00,0.66


In [161]:
questions = ThmQA['train']['Question'][:100]
gold_answers = ThmQA['train']['Answer'][:100]
base_preds = generate_answers(questions, make_base_prompt, Gemma_pipe)
with open('/content/drive/MyDrive/Final Project/src/Results/gemma_thmqa_base_preds.json', 'w') as f:
    json.dump(base_preds, f)
# Run CoT
cot_preds = generate_answers(questions, make_cot_prompt, Gemma_pipe)
with open('/content/drive/MyDrive/Final Project/src/Results/gemma_thmqa_cot_preds.json', 'w') as f:
    json.dump(cot_preds, f)
# Run QAP
qap_preds = generate_answers(questions, make_qap_prompt, Gemma_pipe)
with open('/content/drive/MyDrive/Final Project/src/Results/gemma_thmqa_qap_preds.json', 'w') as f:
    json.dump(qap_preds, f)
# Run MAD
mad_preds = generate_answers(questions, make_mad_prompt, Gemma_pipe)
with open('/content/drive/MyDrive/Final Project/src/Results/gemma_thmqa_mad_preds.json', 'w') as f:
    json.dump(mad_preds, f)
# Evaluate all
base_results = [grade_answer_with_label(pred, gold) for pred, gold in zip(base_preds, gold_answers)]
cot_results = [grade_answer_with_label(pred, gold) for pred, gold in zip(cot_preds, gold_answers)]
qap_results = [grade_answer_with_label(pred, gold) for pred, gold in zip(qap_preds, gold_answers)]
mad_results = [grade_answer_with_label(pred, gold) for pred, gold in zip(mad_preds, gold_answers)]
base_df = pd.DataFrame({
    "method": "Baseline",
    "question": questions,
    "gold": gold_answers,
    "pred": base_preds,
    "eval": base_results
})

cot_df = pd.DataFrame({
    "method": "CoT",
    "question": questions,
    "gold": gold_answers,
    "pred": cot_preds,
    "eval": cot_results
})

qap_df = pd.DataFrame({
    "method": "QAP",
    "question": questions,
    "gold": gold_answers,
    "pred": qap_preds,
    "eval": qap_results
})

mad_df = pd.DataFrame({
    "method": "MAD",
    "question": questions,
    "gold": gold_answers,
    "pred": mad_preds,
    "eval": mad_results
})
all_results = pd.concat([base_df,cot_df, qap_df, mad_df])
accuracy_summary = all_results.groupby("method")["eval"].value_counts(normalize=True).unstack().fillna(0)
from IPython.display import display, Markdown

display(Markdown("### 🔍 Prompting Gemma with ThmQA problems"))
display(accuracy_summary)

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `64` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


### 🔍 Prompting Gemma with ThmQA problems

eval,correct_mathd,correct_semantic,correct_string,incorrect
method,,,,
Baseline,0.09,0.01,0.01,0.89
CoT,0.09,0.03,0.00,0.88
MAD,0.03,0.00,0.00,0.97
QAP,0.07,0.00,0.01,0.92
